<h3>Runnable & Runnable Passthrough</h3>

Runnable is anything you can “pipe” (|) data through (prompts, LLMs, retrievers, functions, etc.).
RunnablePassthrough is a tiny helper Runnable that forwards its input unchanged. It’s most useful when you want to:

* Preserve the original input while also computing extra fields derived from that input.
* Branch: feed the same input to multiple sub-steps (e.g., to an LLM and to a function) and then recombine.
* Enrich: add keys to an object (dict) on-the-fly without losing existing keys.

Think of it as “keep the input as-is, but let me attach more stuff next to it.”
There are two very common patterns:

* RunnablePassthrough() — just forwards input.
* RunnablePassthrough.assign(key1=..., key2=...) — forwards input and adds/overwrites fields using other runnables/functions.

In [ ]:
# Simple passthrough runnable example
from langchain_core.runnables import RunnablePassthrough

pt = RunnablePassthrough()

print(pt.invoke("Hello"))       # -> "Hello"
print(pt.invoke({"x": 1}))      # -> {"x": 1}


In [ ]:
# Example of a more complex runnable that adds 1 to a number
from langchain_core.runnables import Runnable

class AddOneRunnable(Runnable):
    def invoke(self, input):
        return input + 1

add_one = AddOneRunnable()
print(add_one.invoke(5))  # -> 6

In [ ]:
%run langchain_common_notebook.py

In [ ]:
def extract_city(user_text: str) -> str:
    for city in ["Paris", "Riyadh", "Jeddah", "Lahore"]:
        if city.lower() in user_text.lower():
            return city
    return "Unknown"

# Build a small preprocessor that turns the input string into a dict {question: <str>} and enriches it
# output --> {'question': "<prompt>", 'city': '<extracted_city>'}
pre = (
    (lambda x: {"question": x}) # Convert input string to dict with 'question' key
    # Add 'city' while preserving 'question'
    | RunnablePassthrough.assign(city=lambda d: extract_city(d["question"]))
)

pre.invoke("What is the weather in Riyadh?")

In [ ]:
# Build a prompt that expects both 'question' and 'city'
prompt = ChatPromptTemplate.from_template(
    "Answer the user's question.\n"
    "City (detected): {city}\n"
    "Question: {question}"
)


In [ ]:
chain = pre | prompt | llm_noreason 

resp = (chain.invoke("What's the weather in Paris today?").content)

In [ ]:
print(resp)

### We'll fix the above issue with tools, where we can do weather search and bring the results back as part of results!

In [ ]:
# Another RunnablePassthrough example: enrich input with derived fields
import pprintpp


enrich_chain = (
    (lambda text: {"text": text})
    | RunnablePassthrough.assign(
        char_count=lambda d: len(d["text"]),
        word_count=lambda d: len(d["text"].split()),
        is_question=lambda d: d["text"].strip().endswith("?"),
        uppercase=lambda d: d["text"].upper(),
    )
)

result = enrich_chain.invoke("Can RunnablePassthrough enrich this input?")
pprintpp.pprint(result)

## `RunnableSequence` in LangChain

`RunnableSequence` is a LangChain class for chaining multiple runnable steps into one pipeline, where each step’s output becomes the next step’s input.

### What it does
- Composes stages like: **prompt → model → parser**
- Provides a single interface to run the full flow (`invoke`, `batch`, `stream`)
- Helps build clean, reusable LLM workflows (including RAG pipelines)

### In this notebook
Your chain:

- `prompt`
- `llm`
- `StrOutputParser()`

is a `RunnableSequence`, created either with pipe syntax:

```python
rag_chain = prompt | llm | StrOutputParser()
```

or explicitly:

```python
rag_chain = RunnableSequence(prompt, llm, StrOutputParser())
```

Both are equivalent.

### Why it is useful
- Keeps pipeline logic modular
- Makes debugging and swapping components easier
- Works naturally with retrievers and other runnables in LCEL

In [ ]:
rag_chain = RunnableSequence(prompt, llm, StrOutputParser())